# Session Representative Shoreline + Curve Boxplot (per session)

For each session folder (one video clip), this notebook:
1. Loads all `*_warped.csv` shoreline realizations (world/rectified coords).
2. Projects each shoreline to a 1D distance function `d(u)` along a fixed baseline.
3. Computes a functional-depth ordering and outputs:
   - deepest observed representative shoreline
   - pointwise-median representative shoreline (diagnostic)
   - curve boxplot summary (median + central band)

Outputs are written under `OUT_ROOT/<session_name>/`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# -----------------------------
# CONFIG (edit these)
# -----------------------------
# Root folder containing per-session subfolders + a 'baseline' subfolder
BASE_DIR = Path('./csv/seabright_new_rec/')
BASELINE_CSV = BASE_DIR / 'baseline' / 'baseline_warped.csv'

# Each session folder contains many warped shoreline CSVs
SESSION_GLOB = '*_warped.csv'

# Output root (a subfolder per session will be created)
OUT_ROOT = Path('./csv/seabright_new_rec_averaged/')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Resampling along the baseline / alongshore domain
N_S = 400

# Validity filtering
MIN_HIT_FRAC_PER_S = 0.7  # keep alongshore stations where >= this fraction of curves intersect
DROP_CURVES_WITH_NANS = True

# Plot control
SAVE_FIG = True

# Discover session folders
session_dirs = sorted([p for p in BASE_DIR.iterdir() if p.is_dir() and p.name != 'baseline'])
print(f'Found {len(session_dirs)} session folders under: {BASE_DIR}')


In [ ]:
def _pick_xy_columns(df: pd.DataFrame):
    # Prefer warped/world columns
    for xcol, ycol in [('X_warped','Y_warped'), ('x_warped','y_warped'), ('X','Y'), ('x','y')]:
        if xcol in df.columns and ycol in df.columns:
            return xcol, ycol
    raise ValueError(f'No usable XY columns found. Columns={list(df.columns)[:20]}')


def load_polyline_csv(path: Path) -> np.ndarray:
    df = pd.read_csv(path)
    xcol, ycol = _pick_xy_columns(df)
    xy = df[[xcol, ycol]].to_numpy(dtype=float)
    # drop exact duplicate consecutive points
    if len(xy) > 1:
        keep = np.ones(len(xy), dtype=bool)
        keep[1:] = np.any(np.abs(xy[1:] - xy[:-1]) > 0, axis=1)
        xy = xy[keep]
    return xy


def resample_polyline_arclength(xy: np.ndarray, n: int) -> np.ndarray:
    # Uniform arclength resampling
    xy = np.asarray(xy, float)
    if len(xy) == 0:
        return xy
    if len(xy) == 1:
        return np.repeat(xy, n, axis=0)

    seg = xy[1:] - xy[:-1]
    seglen = np.linalg.norm(seg, axis=1)
    s = np.concatenate([[0.0], np.cumsum(seglen)])
    total = s[-1]
    if total == 0:
        return np.repeat(xy[:1], n, axis=0)

    s_new = np.linspace(0.0, total, n)
    x_new = np.interp(s_new, s, xy[:,0])
    y_new = np.interp(s_new, s, xy[:,1])
    return np.column_stack([x_new, y_new])


def compute_normals(baseline_pts: np.ndarray) -> np.ndarray:
    # Unit normals for a baseline polyline (computed via tangents)
    b = np.asarray(baseline_pts, float)
    t = np.zeros_like(b)
    if len(b) == 1:
        return np.array([[0.0, 1.0]])
    t[1:-1] = b[2:] - b[:-2]
    t[0] = b[1] - b[0]
    t[-1] = b[-1] - b[-2]

    t_norm = np.linalg.norm(t, axis=1, keepdims=True)
    t_norm[t_norm == 0] = 1.0
    t = t / t_norm

    # +90deg rotation
    n = np.column_stack([-t[:,1], t[:,0]])
    n_norm = np.linalg.norm(n, axis=1, keepdims=True)
    n_norm[n_norm == 0] = 1.0
    return n / n_norm


def intersect_line_with_segment(o, d, a, b, eps=1e-9):
    # Line: o + t d, t in R; Segment: a + u (b-a), u in [0,1]
    o = np.asarray(o, float)
    d = np.asarray(d, float)
    a = np.asarray(a, float)
    b = np.asarray(b, float)

    v = b - a
    M = np.column_stack((d, -v))
    det = np.linalg.det(M)
    if abs(det) < eps:
        return None

    t, u = np.linalg.solve(M, a - o)
    if -eps <= u <= 1.0 + eps:
        return float(t)
    return None


def distance_curve_to_baseline(shore_xy: np.ndarray, baseline_pts: np.ndarray, normals: np.ndarray) -> np.ndarray:
    # Computes unsigned nearest-intersection distance for each baseline station
    dvals = np.full(len(baseline_pts), np.nan, dtype=float)

    for i, (o, dvec) in enumerate(zip(baseline_pts, normals)):
        hits = []
        for j in range(len(shore_xy) - 1):
            t = intersect_line_with_segment(o, dvec, shore_xy[j], shore_xy[j+1])
            if t is not None:
                hits.append(t)
        if hits:
            t_best = min(hits, key=lambda x: abs(x))
            dvals[i] = abs(t_best)

    return dvals


def fraiman_muniz_depth(curves: np.ndarray) -> np.ndarray:
    # Fraiman-Muniz-style depth for curves array (N, M) without NaNs
    N, M = curves.shape
    # ranks per column
    ranks = np.empty((N, M), dtype=float)
    for j in range(M):
        order = np.argsort(curves[:,j])
        r = np.empty(N, dtype=float)
        r[order] = np.arange(1, N+1)
        ranks[:,j] = r

    F = ranks / (N + 1.0)
    # depth contribution per point: 1 - |2F - 1|
    contrib = 1.0 - np.abs(2.0 * F - 1.0)
    return contrib.mean(axis=1)


In [ ]:
# -----------------------------
# RUN: per-session representative shoreline + curve boxplot
# -----------------------------
# Load and resample baseline once
baseline_raw = load_polyline_csv(BASELINE_CSV)
baseline_pts = resample_polyline_arclength(baseline_raw, N_S)
normals = compute_normals(baseline_pts)

for SESSION_DIR in session_dirs:
    shore_files = sorted([p for p in SESSION_DIR.glob(SESSION_GLOB) if p.name != BASELINE_CSV.name])
    if not shore_files:
        print(f"[skip] {SESSION_DIR.name}: no files matching {SESSION_GLOB}")
        continue

    OUT_DIR = OUT_ROOT / SESSION_DIR.name
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"
[run] {SESSION_DIR.name}: {len(shore_files)} shorelines")

    # Build curves matrix (N_curves, N_S)
    curves = np.full((len(shore_files), N_S), np.nan, dtype=float)
    for k, f in enumerate(shore_files):
        shore_xy = load_polyline_csv(f)
        if len(shore_xy) < 2:
            continue
        curves[k] = distance_curve_to_baseline(shore_xy, baseline_pts, normals)

    # Valid alongshore stations based on hit fraction
    hit_frac = np.mean(~np.isnan(curves), axis=0)
    valid_s = hit_frac >= MIN_HIT_FRAC_PER_S
    n_valid = int(valid_s.sum())
    print(f"Valid alongshore stations: {n_valid} / {N_S}")
    if n_valid == 0:
        print(f"[skip] {SESSION_DIR.name}: no valid stations after filtering")
        continue

    curves_v = curves[:, valid_s]

    if DROP_CURVES_WITH_NANS:
        keep = ~np.any(np.isnan(curves_v), axis=1)
        curves_use = curves_v[keep]
        shore_use_files = [f for f, ok in zip(shore_files, keep) if ok]
        print(f"Curves kept after NaN filtering: {len(shore_use_files)}")
        if len(shore_use_files) == 0:
            print(f"[skip] {SESSION_DIR.name}: all curves had NaNs")
            continue
    else:
        # simple linear fill along s for each curve (rarely used)
        curves_use = curves_v.copy()
        x = np.arange(curves_use.shape[1])
        for i in range(curves_use.shape[0]):
            y = curves_use[i]
            m = ~np.isnan(y)
            if m.sum() < 2:
                continue
            curves_use[i, ~m] = np.interp(x[~m], x[m], y[m])
        shore_use_files = shore_files

    # Depth ordering
    depths = fraiman_muniz_depth(curves_use)
    order = np.argsort(-depths)  # deepest first

    # Deepest observed curve
    idx_deep = int(order[0])
    deep_curve = curves_use[idx_deep]
    deep_file = shore_use_files[idx_deep]

    # Pointwise median curve (diagnostic)
    point_med = np.median(curves_use, axis=0)

    # Central 50% band
    n_central = max(1, len(order)//2)
    central = curves_use[order[:n_central]]
    band_lo = central.min(axis=0)
    band_hi = central.max(axis=0)

    # Map representative curves back to world coords (baseline + d * normal)
    # Build a full-length array over N_S with NaNs on invalid stations
    deep_full = np.full(N_S, np.nan, dtype=float)
    med_full = np.full(N_S, np.nan, dtype=float)
    band_lo_full = np.full(N_S, np.nan, dtype=float)
    band_hi_full = np.full(N_S, np.nan, dtype=float)

    deep_full[valid_s] = deep_curve
    med_full[valid_s] = point_med
    band_lo_full[valid_s] = band_lo
    band_hi_full[valid_s] = band_hi

    # Representative world polylines
    deep_xy = baseline_pts + deep_full[:,None] * normals
    med_xy = baseline_pts + med_full[:,None] * normals

    u = np.linspace(0.0, 1.0, N_S)

    rep_deep_df = pd.DataFrame({'u':u, 'X_warped': deep_xy[:,0], 'Y_warped': deep_xy[:,1]})
    rep_med_df  = pd.DataFrame({'u':u, 'X_warped': med_xy[:,0],  'Y_warped': med_xy[:,1]})

    rep_deep_path = OUT_DIR / f"{SESSION_DIR.name}_representative_deepest.csv"
    rep_med_path  = OUT_DIR / f"{SESSION_DIR.name}_representative_pointwise_median.csv"
    rep_deep_df.to_csv(rep_deep_path, index=False)
    rep_med_df.to_csv(rep_med_path, index=False)

    # Curve boxplot summary
    summary_df = pd.DataFrame({
        'u': u,
        'median_d': deep_full,          # deepest observed curve is the functional median
        'band50_lo': band_lo_full,
        'band50_hi': band_hi_full,
        'hit_frac': hit_frac,
    })
    summary_path = OUT_DIR / f"{SESSION_DIR.name}_curveboxplot_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    # Diagnostic plot
    if SAVE_FIG:
        plt.figure(figsize=(10,4))
        # plot a handful of curves (thin)
        n_plot = min(60, curves_use.shape[0])
        for c in curves_use[np.random.choice(curves_use.shape[0], n_plot, replace=False)]:
            plt.plot(u[valid_s], c, linewidth=0.5, alpha=0.2)
        plt.fill_between(u, band_lo_full, band_hi_full, alpha=0.2, label='central 50% band')
        plt.plot(u, deep_full, linewidth=2.0, label='deepest observed (median)')
        plt.plot(u, med_full, linewidth=1.5, linestyle='--', label='pointwise median (diagnostic)')
        plt.xlabel('u (normalized alongshore)')
        plt.ylabel('distance to baseline (m)')
        plt.title(f"{SESSION_DIR.name} - session curve boxplot")
        plt.legend(loc='best')
        fig_path = OUT_DIR / f"{SESSION_DIR.name}_curveboxplot.png"
        plt.tight_layout()
        plt.savefig(fig_path, dpi=200)
        plt.close()

    print(f"  wrote: {rep_deep_path.name}")
    print(f"  wrote: {rep_med_path.name}")
    print(f"  wrote: {summary_path.name}")
    if SAVE_FIG:
        print(f"  wrote: {fig_path.name}")

print("
Done.")
